# Named Entity Recognition (NER) pada Berita Bencana Indonesia

## 👥 Anggota Kelompok
- Umar Al Faris - 5054241015  
- Muhammad Hadidan Nurhaunan - 5054241016  
- Wayan Raditya Putra - 5054241029  
- Syauqi Nabil Tasri - 5054241040  
- Arvito Rajapandya Natlysandro - 5054241046  

---

## Pendahuluan
Berita bencana di Indonesia mengandung berbagai informasi penting seperti lokasi kejadian, waktu, jenis bencana, organisasi terkait, serta tokoh yang terlibat. Informasi tersebut umumnya tersimpan dalam bentuk teks tidak terstruktur, sehingga sulit untuk dianalisis secara langsung.

Oleh karena itu, diperlukan suatu metode untuk mengekstrak informasi tersebut ke dalam format yang lebih terstruktur dan mudah diproses.

---

## Tujuan
Proyek ini bertujuan untuk mengekstrak entitas penting dari berita bencana, yaitu:
- LOCATION  
- DATE/TIME  
- DISASTER_TYPE  
- ORGANIZATION  
- PERSON  

Pendekatan yang digunakan adalah kombinasi beberapa metode, yaitu:
- Pretrained NER untuk ekstraksi entitas umum  
- Rule-based method untuk identifikasi DISASTER_TYPE  
- Regular expression (regex) untuk ekstraksi DATE/TIME  

---

## Alur Kerja
Pipeline utama yang digunakan dalam proyek ini adalah:

Raw Data → Preprocessing → Pretrained NER → Rule-Based Extraction → Output → Eksperimen Machine Learning

---

## Perangkat yang Digunakan
- Python  
- Pandas  
- Transformers (Hugging Face) dengan model IndoBERT NER  
- Regular Expression (regex) untuk ekstraksi pola teks  
- Scikit-learn untuk eksperimen machine learning  

---

## Dataset
Dataset yang digunakan terdiri dari sekitar 2000 berita bencana di Indonesia dalam format CSV.  

Kolom utama yang digunakan adalah:
- `full_text` sebagai sumber teks utama untuk ekstraksi entitas  

Data kemudian diproses menjadi bentuk teks bersih (`clean_text`) sebelum digunakan dalam pipeline NER.

## Setup
Di bagian ini kita mengimpor library yang dipakai untuk memproses data dan menjalankan model NER.

In [1]:
# Import library

import pandas as pd 
import re
from transformers import pipeline



d:\AI_Journey\NLP\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("berita_bencana.csv")

df.head(5)

,No,judul_berita,author,tanggal,full_text
0,1,Aceh Tetapkan Status Siaga Bencana hingga 20 A...,Feri Agus,2026-04-14 14:19:49,Pemerintah Aceh menetapkan status siaga bencan...
1,2,FAO Waspadai Bencana Pangan Global Imbas Gangg...,Putri Utami,2026-04-14 11:18:38,Organisasi Pangan dan Pertanian ( FAO ) memper...
2,3,Mendagri Tito Sebut Inflasi Bulanan 3 Daerah T...,Indra Hendriana,2026-04-06 15:52:22,Pemerintah menyampaikan bahwa inflasi bulanan ...
3,4,"Gajah Dikerahkan Bantu Bencana, BKSDA Pastikan...",Yugo Hindarto,2025-12-09 12:47:24,Balai Konservasi Sumber Daya Alam ( BKSDA ) Ac...
4,5,Ada 2.463 Titik Rawan Bencana di Indonesia Jel...,Yugo Hindarto,2025-12-19 13:03:57,Sebanyak 2.463 titik wilayah di Indonesia masu...


In [3]:
df.isnull().sum()

No              0
judul_berita    0
author          0
tanggal         0
full_text       0
dtype: int64

## Preprocessing Data

Tahap ini melakukan preprocessing ringan pada teks berita untuk menjaga konsistensi sebelum masuk ke model.

Langkah preprocessing meliputi:
- Lowercase semua teks
- Menghapus spasi berlebih

Preprocessing dibuat minimal karena IndoBERT sudah mampu memahami konteks bahasa Indonesia, dan pembersihan yang terlalu agresif dapat menghilangkan informasi NER penting.

In [4]:
# Preprocess data dengan lower casing dan hapus spasi yang berlebih (jika ada)
df["clean_text"] = df["full_text"].str.lower().str.strip()

df["clean_text"] = df["clean_text"].apply(lambda x: re.sub(r"\s+", " ", x))

# Menampilkan contoh sebelum vs sesudah
print("BEFORE:")
print(df["full_text"].iloc[0])

print("\nAfter:")
print(df["clean_text"].iloc[0])


BEFORE:
Pemerintah Aceh menetapkan status siaga bencana hidrometeorologi di sejumlah kabupaten/kota hingga 20 April 2026. Kebijakan ini diambil menyusul peringatan dini cuaca ekstrem dari Badan Meteorologi, Klimatologi, dan Geofisika (BMKG). Sebelumnya BMKG memperkirakan wilayah Aceh berpotensi dilanda hujan dengan intensitas sedang hingga lebat, disertai angin kencang dan petir dalam beberapa hari ke depan. Kondisi ini dipengaruhi oleh pola siklonik, belokan angin, serta konvergensi yang meningkatkan pertumbuhan awan hujan. Dampaknya hampir seluruh wilayah Aceh berisiko mengalami bencana hidrometeorologi seperti banjir, tanah longsor, dan angin kencang, terutama pada periode 11 hingga 20 April 2026. Sekretaris Daerah Aceh M. Nasir menginstruksikan seluruh pemerintah kabupaten/kota untuk mengaktifkan posko siaga darurat selama 24 jam, khususnya di wilayah rawan bencana. "Kami meminta BPBD di kabupaten/kota mengaktifkan posko dan memantau perkembangan cuaca secara real-time bersama BMKG

## Pendekatan Hybrid NER

Proyek ini menggunakan strategi hybrid antara model pretrained dan rule-based extraction untuk menangani entitas umum sekaligus entitas domain spesifik.

### Pretrained NER
Model pretrained dipakai untuk mengekstrak entitas umum:
- LOCATION
- ORGANIZATION
- PERSON

Model ini membantu mengenali pola entitas dari teks tanpa perlu aturan manual yang kompleks.

### Rule-Based untuk Domain Bencana
Rule-based digunakan untuk mengekstrak entitas yang lebih spesifik domain bencana:
- DISASTER_TYPE
- DATE/TIME

Jenis bencana sering muncul dengan kata kunci yang relatif jelas, sedangkan tanggal/waktu dapat diekstrak menggunakan regex.

### Alur Pipeline

Raw Text → Preprocessing → Pretrained NER → Rule-Based Extraction → Post-processing → Output

Pendekatan ini menggabungkan kekuatan model dan aturan domain untuk hasil yang lebih lengkap.

In [5]:
ner = pipeline(
    "ner",
    model="ageng-anugrah/indobert-large-p2-finetuned-ner",
    aggregation_strategy="simple"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5722.33it/s]
BertForTokenClassification LOAD REPORT from: ageng-anugrah/indobert-large-p2-finetuned-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Chunking dan NER Awal
Teks panjang dibagi menjadi potongan agar pipeline NER dapat memproses seluruh berita secara stabil.

In [7]:
import re

def split_text_by_char(text, max_chars=400):
    chunks = []
    
    # Pecah dulu berdasarkan kalimat
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    current_chunk = ""
    
    for sentence in sentences:
        # Kalau masih muat, tambahkan ke chunk sekarang
        if len(current_chunk) + len(sentence) + 1 <= max_chars:
            current_chunk += " " + sentence
        else:
            # Simpan chunk lama kalau ada isi
            if current_chunk:
                chunks.append(current_chunk.strip())
            # Mulai chunk baru
            current_chunk = sentence
    
    # Tambahkan chunk terakhir
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks


for i in range(5):
    text = df["clean_text"].iloc[i]
    chunks = split_text_by_char(text, max_chars=400)
    
    all_entities = []
    
    for chunk in chunks:
        result = ner(chunk)
        all_entities.extend(result)
    
    print(f"News {i+1}")
    print("Jumlah chunk:", len(chunks))
    print("Text preview:", text[:300])
    print("Entities:", all_entities)
    print("-" * 60)

News 1
Jumlah chunk: 7
Text preview: pemerintah aceh menetapkan status siaga bencana hidrometeorologi di sejumlah kabupaten/kota hingga 20 april 2026. kebijakan ini diambil menyusul peringatan dini cuaca ekstrem dari badan meteorologi, klimatologi, dan geofisika (bmkg). sebelumnya bmkg memperkirakan wilayah aceh berpotensi dilanda huja
Entities: [{'entity_group': 'PLACE', 'score': np.float32(0.99970895), 'word': 'aceh', 'start': 11, 'end': 15}, {'entity_group': 'ORGANISATION', 'score': np.float32(0.99889934), 'word': 'badan meteorologi, klimatologi', 'start': 180, 'end': 210}, {'entity_group': 'ORGANISATION', 'score': np.float32(0.99632955), 'word': 'dan geofisika', 'start': 212, 'end': 225}, {'entity_group': 'ORGANISATION', 'score': np.float32(0.9948782), 'word': 'bmkg', 'start': 227, 'end': 231}, {'entity_group': 'PLACE', 'score': np.float32(0.99978), 'word': 'aceh', 'start': 38, 'end': 42}, {'entity_group': 'PLACE', 'score': np.float32(0.99987876), 'word': 'aceh', 'start': 33, 'end'

In [8]:
label_map = {
    "PLACE": "LOCATION",
    "ORGANISATION": "ORGANIZATION",
    "PERSON": "PERSON"
}

noise_words = {
    "##if", "##ologi,", "gamb", "fa", "pulau", "gedung", "badan"
}

def clean_ner_results(entities):
    cleaned = []
    seen = set()

    for item in entities:
        word = item["word"].strip().lower()
        label = item["entity_group"]
        score = float(item["score"])

        # 1. Skip kalau label tidak ada di mapping
        if label not in label_map:
            continue

        # 2. Skip kalau token aneh / noise
        if "##" in word or word in noise_words:
            continue
        
        if word.startswith("dan "):
            continue

        # 3. Skip kalau terlalu pendek (misal 1 huruf / 2 huruf yang aneh)
        if len(word) <= 2:
            continue

        # 4. Mapping label
        mapped_label = label_map[label]

        # 5. Hindari duplikasi (entity + label)
        key = (word, mapped_label)
        if key in seen:
            continue

        seen.add(key)

        cleaned.append({
            "entity": word,
            "label": mapped_label,
            "score": round(score, 4)
        })

    return cleaned

In [9]:
def remove_partial_duplicates(entities):
    filtered = []

    for i, item in enumerate(entities):
        word_i = item["entity"]
        label_i = item["label"]
        is_partial = False

        for j, other in enumerate(entities):
            if i == j:
                continue

            word_j = other["entity"]
            label_j = other["label"]

            # Kalau label sama dan entity i ada di dalam entity j yang lebih panjang
            if label_i == label_j and word_i in word_j and len(word_i) < len(word_j):
                is_partial = True
                break

        if not is_partial:
            filtered.append(item)

    return filtered

In [10]:
text = df["clean_text"].iloc[0]
chunks = split_text_by_char(text, max_chars=400)

all_entities = []
for chunk in chunks:
    result = ner(chunk)
    all_entities.extend(result)

cleaned_entities = clean_ner_results(all_entities)
final_entities = remove_partial_duplicates(cleaned_entities)

print(final_entities)

[{'entity': 'aceh', 'label': 'LOCATION', 'score': 0.9997}, {'entity': 'badan meteorologi, klimatologi', 'label': 'ORGANIZATION', 'score': 0.9989}, {'entity': 'bmkg', 'label': 'ORGANIZATION', 'score': 0.9949}, {'entity': 'm. nasir', 'label': 'PERSON', 'score': 0.9988}, {'entity': 'bpba', 'label': 'ORGANIZATION', 'score': 0.9997}]


## Rule-Based DISASTER_TYPE
Daftar keyword bencana digunakan untuk mengekstrak jenis bencana yang muncul di teks.

In [11]:
disaster_keywords = [
    "banjir",
    "gempa bumi",
    "gempa",
    "tsunami",
    "tanah longsor",
    "longsor",
    "kebakaran hutan",
    "kebakaran lahan",
    "kebakaran gedung",
    "kebakaran",
    "gunung meletus",
    "letusan gunung",
    "erupsi",
    "kekeringan",
    "abrasi",
    "puting beliung",
    "angin kencang",
    "cuaca ekstrem",
    "bencana hidrometeorologi"
]

In [12]:
def extract_disaster_type(text, disaster_keywords):
    text = text.lower()
    
    found = []

    # cari semua keyword yang muncul
    for disaster in disaster_keywords:
        if disaster in text:
            found.append(disaster)

    # urutkan dari yang paling spesifik / panjang
    found = sorted(found, key=len, reverse=True)

    final = []
    for d in found:
        # simpan hanya jika belum tercakup oleh istilah yang lebih panjang
        if not any(d in longer for longer in final):
            final.append(d)

    return final

## Regex Ekstraksi DATE/TIME
Fungsi ini mendeteksi tanggal, jam, dan hari dalam teks menggunakan pola regex.

In [14]:
import re

def extract_date_time(text):
    text = text.lower()
    
    patterns = [
        r'\b\d{1,2}\s+(?:januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)\s+\d{4}\b',
        r'\b\d{1,2}[/-]\d{1,2}(?:[/-]\d{2,4})?\b',
        r'\b(?:senin|selasa|rabu|kamis|jumat|sabtu|minggu)\b',
        r'\b(?:pukul|jam)\s+\d{1,2}[:.]\d{2}\b',
        r'\b\d{4}-\d{2}-\d{2}\b'
    ]
    
    results = []

    for pattern in patterns:
        matches = re.finditer(pattern, text)
        for match in matches:
            value = match.group().strip()
            if value not in results:
                results.append(value)

    return results

In [15]:
for i in [0, 3, 4]:
    text = df["clean_text"].iloc[i]
    result = extract_date_time(text)

    print(f"News {i+1}")
    print(result)
    print("-" * 30)

News 1
['20 april 2026', '14/4', 'selasa']
------------------------------
News 4
['9/12', 'selasa']
------------------------------
News 5
['29 desember 2025', '10 januari 2026', '8/12', '15/12', 'senin']
------------------------------


## Final Output Aggregation
Contoh berikut menggabungkan entitas NER, jenis bencana, dan tanggal/waktu menjadi struktur hasil akhir.

In [16]:
def build_final_entity_dict(cleaned_ner, disaster_types, date_times):
    final_entities = {
        "LOCATION": [],
        "ORGANIZATION": [],
        "PERSON": [],
        "DISASTER_TYPE": disaster_types,
        "DATE/TIME": date_times
    }

    for item in cleaned_ner:
        label = item["label"]
        entity = item["entity"]

        if label in final_entities and entity not in final_entities[label]:
            final_entities[label].append(entity)

    return final_entities

In [17]:
text = df["clean_text"].iloc[0]

chunks = split_text_by_char(text, max_chars=400)

all_entities = []
for chunk in chunks:
    result = ner(chunk)
    all_entities.extend(result)

cleaned_entities = clean_ner_results(all_entities)
final_ner_entities = remove_partial_duplicates(cleaned_entities)

disaster_result = extract_disaster_type(text, disaster_keywords)
date_time_result = extract_date_time(text)

final_output = build_final_entity_dict(final_ner_entities, disaster_result, date_time_result)

print(final_output)

{'LOCATION': ['aceh'], 'ORGANIZATION': ['badan meteorologi, klimatologi', 'bmkg', 'bpba'], 'PERSON': ['m. nasir'], 'DISASTER_TYPE': ['bencana hidrometeorologi', 'tanah longsor', 'angin kencang', 'cuaca ekstrem', 'banjir'], 'DATE/TIME': ['20 april 2026', '14/4', 'selasa']}


## Eksekusi Pipeline pada Seluruh Dataset
Langkah ini menjalankan pipeline hybrid pada seluruh berita untuk menyimpan hasil NER dan rule-based extraction.

In [18]:
results = []

for i in range(len(df)):
    text = df["clean_text"].iloc[i]
    title = df["judul_berita"].iloc[i]

    try:
        # =========================
        # 1. NER: LOCATION / ORG / PERSON
        # =========================
        chunks = split_text_by_char(text, max_chars=400)

        all_entities = []
        for chunk in chunks:
            try:
                result = ner(chunk)
                all_entities.extend(result)
            except:
                continue

        cleaned_entities = clean_ner_results(all_entities)
        final_ner_entities = remove_partial_duplicates(cleaned_entities)

        # =========================
        # 2. RULE: DISASTER_TYPE
        # =========================
        disaster_result = extract_disaster_type(text, disaster_keywords)

        # =========================
        # 3. REGEX: DATE/TIME
        # =========================
        date_time_result = extract_date_time(text)

        # =========================
        # 4. COMBINE
        # =========================
        final_output = build_final_entity_dict(
            final_ner_entities,
            disaster_result,
            date_time_result
        )

        # =========================
        # 5. SAVE RESULT
        # =========================
        results.append({
            "id": i,
            "judul_berita": title,
            "LOCATION": ", ".join(final_output["LOCATION"]) if final_output["LOCATION"] else "-",
            "ORGANIZATION": ", ".join(final_output["ORGANIZATION"]) if final_output["ORGANIZATION"] else "-",
            "PERSON": ", ".join(final_output["PERSON"]) if final_output["PERSON"] else "-",
            "DISASTER_TYPE": ", ".join(final_output["DISASTER_TYPE"]) if final_output["DISASTER_TYPE"] else "-",
            "DATE/TIME": ", ".join(final_output["DATE/TIME"]) if final_output["DATE/TIME"] else "-"
        })

    except Exception as e:
        print(f"Error di baris {i}: {e}")

        results.append({
            "id": i,
            "judul_berita": title,
            "LOCATION": "-",
            "ORGANIZATION": "-",
            "PERSON": "-",
            "DISASTER_TYPE": "-",
            "DATE/TIME": "-"
        })

    # Progress monitor
    if (i + 1) % 50 == 0:
        print(f"Sudah memproses {i+1} data...")

    # Checkpoint tiap 100 data
    if (i + 1) % 100 == 0:
        temp_df = pd.DataFrame(results)
        temp_df.to_csv("ner_results_checkpoint.csv", index=False)
        print(f"Checkpoint saved at {i+1} data")

Sudah memproses 50 data...
Sudah memproses 100 data...
Checkpoint saved at 100 data
Sudah memproses 150 data...
Sudah memproses 200 data...
Checkpoint saved at 200 data
Sudah memproses 250 data...
Sudah memproses 300 data...
Checkpoint saved at 300 data
Sudah memproses 350 data...
Sudah memproses 400 data...
Checkpoint saved at 400 data
Sudah memproses 450 data...
Sudah memproses 500 data...
Checkpoint saved at 500 data
Sudah memproses 550 data...
Sudah memproses 600 data...
Checkpoint saved at 600 data
Sudah memproses 650 data...
Sudah memproses 700 data...
Checkpoint saved at 700 data
Sudah memproses 750 data...
Sudah memproses 800 data...
Checkpoint saved at 800 data
Sudah memproses 850 data...
Sudah memproses 900 data...
Checkpoint saved at 900 data
Sudah memproses 950 data...
Sudah memproses 1000 data...
Checkpoint saved at 1000 data
Sudah memproses 1050 data...
Sudah memproses 1100 data...
Checkpoint saved at 1100 data
Sudah memproses 1150 data...
Sudah memproses 1200 data...
Che

In [19]:
output_df = pd.DataFrame(results)
output_df.to_csv("ner_results_2000.csv", index=False)
print("Selesai. File disimpan sebagai ner_results_2000.csv")

Selesai. File disimpan sebagai ner_results_2000.csv


## Eksperimen dengan Normalisasi Label

Pada tahap ini dilakukan eksperimen lanjutan dengan tujuan meningkatkan performa model klasifikasi untuk entitas **DISASTER_TYPE**. Eksperimen difokuskan pada dua aspek utama, yaitu normalisasi label dan penyesuaian distribusi kelas.

---

### Normalisasi Label

Label hasil ekstraksi awal memiliki variasi yang berbeda untuk makna yang sama, seperti:
- "longsor" dan "tanah longsor"
- "gempa" dan "gempa bumi"
- "erupsi", "gunung meletus", dan "letusan gunung"
- berbagai jenis "kebakaran"

Variasi ini dapat membingungkan model karena dianggap sebagai kelas yang berbeda. Oleh karena itu, dilakukan normalisasi label untuk menyatukan istilah-istilah yang memiliki makna serupa ke dalam satu kategori yang lebih konsisten.

Proses normalisasi dilakukan menggunakan fungsi `normalize_label`, yang memetakan beberapa label ke bentuk representatifnya.

---

### Penyaringan Data

Dataset kemudian dipersiapkan dengan langkah berikut:
- Menggunakan hasil ekstraksi `DISASTER_TYPE` sebagai label
- Menghapus data yang tidak memiliki label (`"-"`)
- Mengambil satu label utama dari setiap data (single-label classification)

Selanjutnya, dilakukan penyaringan terhadap kelas yang memiliki jumlah data terlalu sedikit. Kelas dengan jumlah kurang dari 10 data dihapus untuk menghindari ketidakseimbangan ekstrem yang dapat menurunkan performa model.

---

### Representasi Fitur

Teks diubah menjadi representasi numerik menggunakan **TF-IDF (Term Frequency-Inverse Document Frequency)**.

Konfigurasi yang digunakan:
- `max_features=5000` untuk membatasi jumlah fitur
- `ngram_range=(1, 2)` untuk menangkap unigram dan bigram

Penggunaan bigram membantu model mengenali frasa penting seperti:
- "tanah longsor"
- "gempa bumi"
- "cuaca ekstrem"

---

### Pelatihan Model

Model yang digunakan adalah **Logistic Regression**, dengan parameter:
- `max_iter=1000` untuk memastikan konvergensi
- `class_weight="balanced"` untuk mengatasi ketidakseimbangan data

Penggunaan `class_weight` membantu model memberikan perhatian lebih pada kelas minoritas.

---

### Evaluasi Model

Model dievaluasi menggunakan:
- Accuracy
- Precision
- Recall
- F1-score

Evaluasi dilakukan pada data uji (test set) yang diperoleh dari pembagian dataset dengan rasio 80:20 menggunakan stratified split.

Parameter `zero_division=0` digunakan untuk menghindari error pada kelas yang tidak memiliki prediksi.

---

### Tujuan Eksperimen

Eksperimen ini bertujuan untuk:
- Meningkatkan konsistensi label
- Mengurangi noise akibat label yang tidak seragam
- Mengatasi ketidakseimbangan data
- Mengevaluasi performa model klasifikasi berbasis teks sederhana

Hasil dari eksperimen ini kemudian digunakan untuk membandingkan efektivitas pendekatan machine learning terhadap hasil ekstraksi sebelumnya.

In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

def normalize_label(label):
    label = label.strip().lower()

    if label in ["longsor", "tanah longsor"]:
        return "longsor"
    elif label in ["gempa", "gempa bumi"]:
        return "gempa bumi"
    elif label in ["gunung meletus", "letusan gunung", "erupsi"]:
        return "erupsi"
    elif label in ["kebakaran", "kebakaran hutan", "kebakaran gedung", "kebakaran lahan"]:
        return "kebakaran"
    else:
        return label

# load hasil NER kalau belum ada
output_df = pd.read_csv("ner_results_2000.csv")

# siapkan dataset ML
ml_df = df[["clean_text"]].copy()
ml_df["DISASTER_TYPE"] = output_df["DISASTER_TYPE"]

# buang data tanpa disaster label
ml_df = ml_df[ml_df["DISASTER_TYPE"] != "-"].copy()
ml_df = ml_df.reset_index(drop=True)

# ambil label pertama
ml_df["label"] = ml_df["DISASTER_TYPE"].apply(lambda x: x.split(",")[0].strip())

# normalisasi label
ml_df["label"] = ml_df["label"].apply(normalize_label)

print("Distribusi label setelah normalisasi:")
print(ml_df["label"].value_counts())

# buang class terlalu kecil
counts = ml_df["label"].value_counts()
valid_labels = counts[counts >= 10].index

ml_df = ml_df[ml_df["label"].isin(valid_labels)].copy()
ml_df = ml_df.reset_index(drop=True)

print("\nDistribusi label setelah buang class kecil:")
print(ml_df["label"].value_counts())

# fitur dan target
X = ml_df["clean_text"]
y = ml_df["label"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# model
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train_vec, y_train)

# evaluasi
y_pred = model.predict(X_test_vec)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Distribusi label setelah normalisasi:
label
longsor                     851
banjir                      285
bencana hidrometeorologi    261
gempa bumi                   56
cuaca ekstrem                52
kebakaran                    44
tsunami                      40
angin kencang                36
erupsi                       21
puting beliung               12
kekeringan                    7
Name: count, dtype: int64

Distribusi label setelah buang class kecil:
label
longsor                     851
banjir                      285
bencana hidrometeorologi    261
gempa bumi                   56
cuaca ekstrem                52
kebakaran                    44
tsunami                      40
angin kencang                36
erupsi                       21
puting beliung               12
Name: count, dtype: int64

Accuracy: 0.7168674698795181

Classification Report:
                          precision    recall  f1-score   support

           angin kencang       0.44      0.57      0.50     

## Analisis Hasil Eksperimen

Setelah dilakukan normalisasi label, terlihat bahwa jumlah kelas menjadi lebih terstruktur dan representatif. Beberapa label yang sebelumnya terpisah berhasil digabungkan, sehingga distribusi data menjadi lebih seimbang. 

Kelas dominan seperti **longsor** memiliki jumlah data yang jauh lebih besar dibandingkan kelas lainnya, sementara kelas minoritas seperti **erupsi** dan **puting beliung** memiliki jumlah yang relatif kecil. Untuk mengurangi dampak ketidakseimbangan ini, kelas dengan jumlah data yang sangat sedikit dihapus dari dataset.

---

### Distribusi Label

Distribusi label setelah normalisasi menunjukkan bahwa:
- Kelas **longsor** menjadi kelas dengan jumlah data terbesar
- Kelas seperti **banjir** dan **bencana hidrometeorologi** juga memiliki jumlah data yang cukup signifikan
- Kelas lain seperti **erupsi**, **tsunami**, dan **angin kencang** masih tergolong minoritas

Setelah dilakukan penyaringan, kelas dengan jumlah data kurang dari 10 dihapus untuk menjaga stabilitas model. Hal ini membantu mengurangi noise dan meningkatkan kemampuan model dalam melakukan generalisasi.

---

### Evaluasi Model

Model Logistic Regression yang digunakan menghasilkan performa sebagai berikut:

- **Accuracy**: 0.72  
- **Macro F1-score**: 0.64  
- **Weighted F1-score**: 0.72  

Hasil ini menunjukkan bahwa model mampu melakukan klasifikasi dengan cukup baik, terutama pada kelas-kelas dominan.

---

### Analisis Performa per Kelas

Beberapa temuan penting dari hasil evaluasi:

- Kelas **longsor** memiliki performa terbaik dengan F1-score sebesar 0.82, menunjukkan bahwa model mampu mengenali pola dari kelas dominan dengan baik.
- Kelas **gempa bumi** dan **tsunami** juga menunjukkan performa yang cukup tinggi dengan precision dan recall yang seimbang.
- Kelas **banjir** dan **bencana hidrometeorologi** memiliki performa yang stabil, meskipun masih terdapat kesalahan klasifikasi.
- Kelas minoritas seperti **cuaca ekstrem** dan **angin kencang** memiliki performa yang lebih rendah, menunjukkan bahwa model masih kesulitan dalam mengenali kelas dengan jumlah data terbatas.

Perbedaan antara **macro average** dan **weighted average** menunjukkan adanya ketidakseimbangan data. Nilai weighted F1-score yang lebih tinggi menandakan bahwa model lebih baik pada kelas dengan jumlah data besar, sementara performa pada kelas kecil masih terbatas.

---

## Kesimpulan

Eksperimen machine learning menggunakan TF-IDF dan Logistic Regression menunjukkan bahwa model sederhana dapat mencapai performa yang cukup baik untuk klasifikasi jenis bencana.

Normalisasi label terbukti efektif dalam:
- Mengurangi variasi label yang tidak konsisten
- Meningkatkan jumlah data per kelas
- Membantu model dalam mempelajari pola yang lebih jelas

Namun, hasil juga menunjukkan bahwa ketidakseimbangan data masih menjadi tantangan utama, terutama untuk kelas dengan jumlah data yang kecil.

Secara keseluruhan, pendekatan ini berhasil menunjukkan bahwa metode machine learning sederhana dapat digunakan sebagai baseline untuk klasifikasi teks, meskipun masih memiliki keterbatasan dibandingkan pendekatan berbasis aturan dan model pretrained yang lebih kompleks.